# 07 — Challengers

In [1]:
# Cell 1 — setup
import sys, pathlib
sys.path.append(str(pathlib.Path.cwd().parent / "src"))
from scorelab import config, data
con = data.connect()            # read-only DuckDB connection to data/lc.duckdb

In [2]:
# Cell 2 — sanity check
print(con.sql("SHOW TABLES"))
print(con.sql("SELECT COUNT(*) AS n, MIN(issue_d), MAX(issue_d) FROM accepted"))

┌─────────────────┐
│      name       │
│     varchar     │
├─────────────────┤
│ accepted        │
│ rejected        │
│ v_accepted_asof │
│ v_split         │
│ v_target        │
└─────────────────┘

┌─────────┬──────────────┬──────────────┐
│    n    │ min(issue_d) │ max(issue_d) │
│  int64  │     date     │     date     │
├─────────┼──────────────┼──────────────┤
│ 2260668 │ 2007-06-01   │ 2018-12-01   │
└─────────┴──────────────┴──────────────┘



In [3]:
# Cell 3 — common imports and settings for this notebook
import json, warnings; warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, matplotlib.pyplot as plt
pd.set_option("display.width", 180); pd.set_option("display.max_columns", 40); pd.set_option("display.max_rows", 200)
from scorelab import target, attributes as A, binning as B, scorecard as S, validation as V, fairness as F
from scorelab import challengers as C, reject_inference as R, monitoring as M, report as REP
MODELS = config.MODELS_DIR; MODELS.mkdir(exist_ok=True)
SAMPLE_FRAC = None     # None = full data (recommended for the fitting notebooks); e.g. 0.10 for a quick pass
cols = [r[0] for r in con.execute("DESCRIBE v_target").fetchall()]
assert "label" in cols, "v_target is the old stub: close this kernel and run `python scripts/refresh_views.py`"
def load_frame(splits=("train", "validate", "oot"), **kw):
    """Observable, non-excluded loans with attributes built; indeterminates dropped unless asked."""
    f = target.modeling_frame(con, splits=splits, sample_frac=SAMPLE_FRAC, **kw)
    return A.build_attributes(f)
MEANINGS = {a.name: a.meaning for a in A.REGISTRY}
DIRECTIONS = {a.name: a.direction for a in A.REGISTRY}

## What this notebook settles

Whether a constrained gradient-boosting challenger beats the card out of time by enough to matter, what it finds that the card does not (interactions, finer thresholds), and whether that signal can be translated back into attributes the card can carry — the ML-to-compliant-attributes method. The challenger descends the same Bernoulli log-loss; it differs in function class and search.

Decisions: **N — constraints and hyper-parameters**, **O — which GBM thresholds become card attributes**, **P — the champion decision memo**.

In [4]:
frame = load_frame()
tr, va, oot = (frame[frame.split == s] for s in ("train", "validate", "oot"))
card, specs = S.load_card(MODELS / "champion"); card_attrs = card.attributes
cands = json.loads((MODELS / "candidate_attributes.json").read_text(encoding="utf-8"))["candidates"]
sc_oot = S.score(oot, card, specs); gini_card = V.delong_ci(oot["bad"], sc_oot["score"])
print("card OOT gini %.4f [%.4f, %.4f] on %d attributes" % (gini_card["gini"], *gini_card["gini_ci"], len(card_attrs)))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

card OOT gini 0.2838 [0.2788, 0.2888] on 8 attributes


### Decision N — Constraints and hyper-parameters

Monotone constraints from the registry are on by default (required for reason codes). Depth, minimum child weight and learning rate are the capacity dials; early stopping is on the validation vintage, never on OOT. Consider interaction constraints that keep attributes of one family together.

**Your reasoning:**

**DRAFT** (Claude, 2026-11-01, from this notebook's full-data outputs; for review). The notebook ran with the defaults in the code cells; this draft says whether to keep them.

**Keep the monotone constraints from the registry and the default capacity** (`max_depth` 4, `min_child_weight` 50, `eta` 0.05, early stopping on 2015).

- **The XGBoost on all 27 candidates** reaches a 2016 Gini of 0.3046 [0.2997, 0.3095] at 304 trees. That is 0.0208 above the card (DeLong 95% CI 0.0188 to 0.0228, z = 20.0).
- **LightGBM matches it** (0.3047), so the lift is not specific to one library.
- **On the card's own 8 attributes the GBM reaches only 0.2910** (+0.0072). About two thirds of the lift (0.0136 of 0.0208) therefore comes from the wider input set, and one third from the model form.
- **No interaction constraints are proposed.** Interaction carries 1–7% of each attribute's SHAP variance (the largest are `revol_bal` 7.4% and `bc_open_to_buy` 7.3%). The model is close to additive already.

In [5]:
XGB_PARAMS = {"max_depth": 4, "min_child_weight": 50, "eta": 0.05, "subsample": 0.8, "colsample_bytree": 0.8}
xgb_all = C.fit_xgb(tr, tr["bad"], va, va["bad"], cands, monotone=True, params=XGB_PARAMS)
xgb_card = C.fit_xgb(tr, tr["bad"], va, va["bad"], card_attrs, monotone=True, params=XGB_PARAMS)
lgb_all = C.fit_lgbm(tr, tr["bad"], va, va["bad"], cands, monotone=True)
rows = []
for name, model, pred in (("xgb, all candidates", xgb_all, C.predict_xgb), ("xgb, card attributes only", xgb_card, C.predict_xgb), ("lgbm, all candidates", lgb_all, C.predict_lgbm)):
    p = pred(model, oot); m = V.delong_ci(oot["bad"], p, higher_is_safer=False)
    rows.append({"model": name, "best_iteration": int(model.best_iteration), "gini_oot": round(m["gini"], 4), "gini_ci": tuple(round(x, 4) for x in m["gini_ci"]),
                 "lift_vs_card": round(m["gini"] - gini_card["gini"], 4)})
rows.append({"model": "champion card", "best_iteration": None, "gini_oot": round(gini_card["gini"], 4), "gini_ci": tuple(round(x, 4) for x in gini_card["gini_ci"]), "lift_vs_card": 0.0})
display(pd.DataFrame(rows))
p_xgb = C.predict_xgb(xgb_all, oot); display(pd.Series(V.delong_test(oot["bad"], -p_xgb, sc_oot["score"])))

,model,best_iteration,gini_oot,gini_ci,lift_vs_card
0,"xgb, all candidates",304.0,0.3046,"(0.2997, 0.3095)",0.0208
1,"xgb, card attributes only",154.0,0.2910,"(0.2861, 0.296)",0.0072
2,"lgbm, all candidates",221.0,0.3047,"(0.2998, 0.3097)",0.0209
3,champion card,NaN,0.2838,"(0.2788, 0.2888)",0.0000


auc_1                        0.6523
auc_2                      0.641903
gini_diff                  0.020793
gini_diff_se               0.001039
gini_diff_ci    (0.018757, 0.02283)
z                          20.01242
p_value                         0.0
dtype: object

In [6]:
# Evidence — what the GBM uses, and whether its reasons agree with the card's
sv, bias = C.shap_values(xgb_all, oot, max_rows=20000)
imp = C.shap_importance(sv); display(imp.head(20))
imp.to_csv(config.DOCS_DIR / "exhibits" / "shap_importance_xgb.csv", index=False)
sample = oot.loc[sv.index[:2000]]
rc_card = S.reason_codes(sample, card, specs, meanings=MEANINGS, missing_reasons=A.MISSING_REASONS); rc_gbm = C.gbm_reason_codes(sv.loc[sample.index], meanings=MEANINGS)
agree = np.mean([len({rc_card.iloc[i, j].split(" [")[0] for j in range(4)} & set(rc_gbm.iloc[i, :4])) for i in range(len(sample))])
print(f"mean overlap of top-4 reason codes, card vs GBM: {agree:.2f} of 4")

,attribute,mean_abs_shap,share_of_total
0,new_trades_share_24m,0.1503,0.1117
1,percent_bc_gt_75,0.1222,0.0908
2,mths_since_recent_inq,0.1117,0.0830
3,tot_hi_cred_lim,0.0962,0.0715
4,num_tl_op_past_12m,0.0847,0.0629
5,bc_open_to_buy,0.0844,0.0627
6,acc_open_past_24mths,0.0800,0.0594
7,bal_ex_mort_to_limit,0.0784,0.0583
8,mo_sin_old_rev_tl_op,0.0706,0.0524
9,revol_bal,0.0590,0.0439


mean overlap of top-4 reason codes, card vs GBM: 1.99 of 4


In [7]:
# Evidence — how much of the GBM is interaction (what an additive card cannot carry), and a surrogate tree
inter = C.interaction_strength(xgb_all, cands, oot, max_rows=3000); display(inter.head(15))
tree, fidelity = C.surrogate_tree(oot, p_xgb, cands, max_depth=3)
print(f"surrogate tree fidelity R2 = {fidelity:.3f}"); print(C.surrogate_rules(tree, cands))

,attribute,main_effect_var,interaction_var,interaction_share
24,new_trades_share_24m,0.02928,0.00041,0.0138
11,percent_bc_gt_75,0.01740,0.00110,0.0596
1,mths_since_recent_inq,0.01629,0.00073,0.0431
18,tot_hi_cred_lim,0.01502,0.00039,0.0251
5,num_tl_op_past_12m,0.01119,0.00018,0.0156
12,bc_open_to_buy,0.01002,0.00079,0.0727
26,bal_ex_mort_to_limit,0.00860,0.00029,0.0324
4,acc_open_past_24mths,0.00852,0.00014,0.0156
19,mo_sin_old_rev_tl_op,0.00697,0.00025,0.0343
13,revol_bal,0.00569,0.00045,0.0740


surrogate tree fidelity R2 = 0.508
|--- new_trades_share_24m <= 0.22
|   |--- bc_open_to_buy <= 7350.50
|   |   |--- new_trades_share_24m <= 0.13
|   |   |   |--- value: [0.07]
|   |   |--- new_trades_share_24m >  0.13
|   |   |   |--- value: [0.10]
|   |--- bc_open_to_buy >  7350.50
|   |   |--- inq_last_6mths <= 0.50
|   |   |   |--- value: [0.05]
|   |   |--- inq_last_6mths >  0.50
|   |   |   |--- value: [0.07]
|--- new_trades_share_24m >  0.22
|   |--- bal_ex_mort_to_limit <= 0.51
|   |   |--- inq_last_6mths <= 1.50
|   |   |   |--- value: [0.10]
|   |   |--- inq_last_6mths >  1.50
|   |   |   |--- value: [0.15]
|   |--- bal_ex_mort_to_limit >  0.51
|   |   |--- num_tl_op_past_12m <= 3.50
|   |   |   |--- value: [0.14]
|   |   |--- num_tl_op_past_12m >  3.50
|   |   |   |--- value: [0.18]



### Decision O — Thresholds that become attributes

The GBM's highest-gain splits are candidate bin edges or flags the card did not have. Choose which to carry back: re-bin that attribute with the threshold as a forced edge (NB04 `OVERRIDES`), or add the flag as a new attribute, then refit the card and measure the Gini recovered.

**Your reasoning:**

**DRAFT** (Claude, 2026-11-01, from this notebook's full-data outputs; for review). The notebook ran with the defaults in the code cells; this draft says whether to keep them.

**Do not change the champion card in this stage. Record the two flags that entered as the first option for a card v2.**

- **Offered:** the 10 highest-gain GBM splits, as flags.
- **Entered the stepwise:** two, both with the right sign:
  - `acc_open_past_24mths_gt_5` (coefficient −0.318, p < 1e-22);
  - `inq_last_6mths_gt_2` (−0.496, p < 1e-26).
- **Gain:** the 10-attribute card reaches 0.2895 [0.2846, 0.2945] on 2016, against 0.2838. It recovers 0.0057 of the GBM's 0.0208 lift, about 27%.
- **Five of the 10 candidates are thresholds on `new_trades_share_24m`** (0.171 to 0.237). None entered, because the card's binning of that attribute already carries them.
- **Why wait:** taking the flags means re-running NB04–NB08. The gain is real but small next to the gap to the GBM, so it is a decision for after the pilot, not inside it.

In [8]:
cand_flags = C.splits_to_attributes(xgb_all, tr, top_k=10); display(cand_flags)
KEEP_FLAGS = cand_flags.candidate.tolist()[:5]         # edit
tr2, va2, oot2 = (C.add_split_flags(d, cand_flags[cand_flags.candidate.isin(KEEP_FLAGS)]) for d in (tr, va, oot))
specs2 = dict(specs)
for f in KEEP_FLAGS:
    specs2[f] = B.fit_numeric_with_edges(tr2[f], tr2["bad"], [0.5], name=f)
W2 = B.transform_woe(tr2, specs2)
sel2, log2 = S.stepwise_select(W2, tr2["bad"], card_attrs + KEEP_FLAGS, p_enter=0.01, max_corr=0.7, max_attributes=20)
card2 = S.fit_logistic(W2, tr2["bad"], sel2)
g2 = V.delong_ci(oot2["bad"], S.score(oot2, card2, specs2)["score"])
print("card + GBM flags: OOT gini %.4f [%.4f, %.4f] with %d attributes (was %.4f)" % (g2["gini"], *g2["gini_ci"], len(sel2), gini_card["gini"]))
display(S.coefficient_table(card2))

,candidate,attribute,threshold,total_gain,share_above
0,acc_open_past_24mths_gt_5,acc_open_past_24mths,5.000,2765.80,0.2750
1,num_tl_op_past_12m_gt_2,num_tl_op_past_12m,2.000,2425.93,0.3090
2,inq_last_6mths_gt_2,inq_last_6mths,2.000,2351.02,0.0723
3,new_trades_share_24m_gt_0.204,new_trades_share_24m,0.204,2009.86,0.3413
4,mths_since_recent_inq_gt_3,mths_since_recent_inq,3.000,1960.40,0.5766
5,new_trades_share_24m_gt_0.194,new_trades_share_24m,0.194,1818.46,0.3728
6,new_trades_share_24m_gt_0.171,new_trades_share_24m,0.171,1758.59,0.4510
7,num_tl_op_past_12m_gt_3,num_tl_op_past_12m,3.000,1734.19,0.1446
8,new_trades_share_24m_gt_0.237,new_trades_share_24m,0.237,1505.00,0.2573
9,new_trades_share_24m_gt_0.217,new_trades_share_24m,0.217,1428.01,0.3089


card + GBM flags: OOT gini 0.2895 [0.2846, 0.2945] with 10 attributes (was 0.2838)


,attribute,coef,std_err,p_value,right_sign
0,new_trades_share_24m,-0.4411,0.0239,3.524809e-76,True
1,num_tl_op_past_12m,-0.5322,0.0286,2.780923e-77,True
2,bal_ex_mort_to_limit,-0.3888,0.0314,3.778145e-35,True
3,bc_open_to_buy,-0.4145,0.0332,1.042466e-35,True
4,tot_hi_cred_lim,-0.6365,0.0350,6.010910e-74,True
5,mths_since_recent_inq,-0.6957,0.0293,2.267832e-124,True
6,mo_sin_old_rev_tl_op,-0.4044,0.0316,1.698824e-37,True
7,percent_bc_gt_75,-0.9090,0.0442,4.728175e-94,True
8,acc_open_past_24mths_gt_5,-0.3177,0.0322,5.114055e-23,True
9,inq_last_6mths_gt_2,-0.4964,0.0463,7.267703e-27,True


### Decision P — Champion decision memo

Write the memo here: the lift with its interval at the out-of-time level, what the GBM found, what the card recovered after translation, the explainability and stability cost of each option, and the decision (keep the card, keep the card with translated attributes, or promote the GBM with SHAP reason codes).

**Your memo:**

**DRAFT** (Claude, 2026-11-01, from this notebook's full-data outputs; for review). The notebook ran with the defaults in the code cells; this draft says whether to keep them.

**Memo (draft). Recommendation: keep the scorecard as champion for the pilot.**

**Lift.** On the 2016 out-of-time vintage the GBM on all 27 candidates beats the card by 0.0208 Gini (0.3046 against 0.2838; DeLong 95% CI 0.0188 to 0.0228). The lift is statistically clear but modest in absolute terms.

**What the GBM found.**
- Most of the lift comes from inputs the card leaves out (on the card's 8 attributes the GBM gains only 0.0072).
- It is not interactions: interaction is 1–7% of SHAP variance per attribute.
- A depth-3 surrogate tree explains 51% of the GBM's predictions (R² 0.508) with `new_trades_share_24m`, `bc_open_to_buy`, `bal_ex_mort_to_limit`, inquiries and trades opened.

**What the card recovered.** Adding two GBM flags lifts the card to 0.2895, about 27% of the gap (Decision O).

**Cost of each option.**
- **Card:** points-based reasons, monotone by construction, already validated across three levels.
- **Card with flags:** about +0.006 Gini for two more attributes and a re-run of NB04–NB08.
- **GBM with SHAP reasons:** the full +0.021. But its top-4 reasons overlap the card's in 1.99 of 4 on average, so a borrower would often hear a different story. It also needs SHAP reason-code governance that the card does not.

**Decision:** keep the card for the pilot; carry the flags as the v2 option; keep the GBM as the benchmark that bounds what the card leaves on the table.

In [9]:
xgb_all.save_model(str(MODELS / "challenger_xgb.json"))
(MODELS / "challenger_decisions.json").write_text(json.dumps({"xgb_params": XGB_PARAMS, "candidates": cands, "kept_flags": KEEP_FLAGS, "gini_card_oot": gini_card["gini"],
           "gini_card_plus_flags_oot": g2["gini"]}, indent=1), encoding="utf-8")
print("saved", (MODELS / "challenger_xgb.json").relative_to(config.ROOT).as_posix())

saved models/challenger_xgb.json
